## Current Progress

- Cleaned dataset
- Features selected
- Isolation forest trained
- Predictions generated
- 339 or 5% of dataset were anomalies

## To do
- Compare predictions with label, attack_type
- Evaluate model performance(IIMT 2641)

In [42]:
import pandas as pd
df=pd.read_csv("../data/cybersecurity_cleaned.csv")
df

,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,bytes_sent,bytes_received,user_agent,url,is_internal_traffic,label,attack_type
0,2025-10-01 00:12:54,18817627165,253240113218,56377,445,0,8029,17204,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/login?id=385071,0,0,benign
1,2025-10-01 00:23:43,68592643,2127538111,51165,1433,0,676368,2643374,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/owa/auth/logon.aspx...,0,0,benign
2,2025-10-01 00:27:21,122119194175,17514078230,36097,443,0,70933,21935,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/phpmyadmin?id=114701,0,0,benign
3,2025-10-01 00:40:09,18119924268,559917769,445,21255,0,12721,9939,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/config.php?id=345569,0,0,benign
4,2025-10-01 01:19:57,238229195174,161827224,25167,443,0,3592,1081,Mozilla/5.0 (iPhone; CPU iPhone OS 16_6 like M...,https://internal.bank.local/?id=432022,1,0,benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6763,2025-12-29 22:35:29,1003151108,2491798177,44655,3306,0,102224,161909,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://webmail.corp/admin?id=667076,0,0,benign
6764,2025-12-29 22:51:57,152953114,236104126125,3306,22,1,244511,280085,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/phpmyadmin?id=120556,1,0,benign
6765,2025-12-29 23:03:05,21354194229,2163437197,3389,445,0,10597,16670,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://app.company.in/api/v1/users?id=525850,0,0,benign
6766,2025-12-29 23:20:46,1912213100,243249117224,31882,80,0,6062,3504,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,https://app.company.in/manager/html?id=497838,0,0,benign


In [43]:
df["timestamp"]=df["timestamp"].astype("datetime64[s]")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6768 entries, 0 to 6767
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype        
---  ------               --------------  -----        
 0   timestamp            6768 non-null   datetime64[s]
 1   src_ip               6768 non-null   int64        
 2   dst_ip               6768 non-null   int64        
 3   src_port             6768 non-null   int64        
 4   dst_port             6768 non-null   int64        
 5   protocol             6768 non-null   int64        
 6   bytes_sent           6768 non-null   int64        
 7   bytes_received       6768 non-null   int64        
 8   user_agent           6768 non-null   str          
 9   url                  6768 non-null   str          
 10  is_internal_traffic  6768 non-null   int64        
 11  label                6768 non-null   int64        
 12  attack_type          6768 non-null   str          
dtypes: datetime64[s](1), int64(9), str(3)
memory usage: 687.5 K

In [44]:
features = ["src_port", "dst_port", "protocol", "bytes_sent", "bytes_received", "is_internal_traffic"]
X = df[features]

In [45]:
from sklearn.ensemble import IsolationForest
iso_forest = IsolationForest(contamination=0.05)
iso_forest.fit(X)
predictions = iso_forest.predict(X)
predictions

array([ 1, -1,  1, ...,  1,  1,  1], shape=(6768,))

In [46]:
#to answer how many anomalies did the model find?
count = 0
for i in predictions:
    if i==-1:
        count += 1
count

339

In [47]:
df["prediction"] = predictions
df

,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,bytes_sent,bytes_received,user_agent,url,is_internal_traffic,label,attack_type,prediction
0,2025-10-01 00:12:54,18817627165,253240113218,56377,445,0,8029,17204,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/login?id=385071,0,0,benign,1
1,2025-10-01 00:23:43,68592643,2127538111,51165,1433,0,676368,2643374,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/owa/auth/logon.aspx...,0,0,benign,-1
2,2025-10-01 00:27:21,122119194175,17514078230,36097,443,0,70933,21935,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/phpmyadmin?id=114701,0,0,benign,1
3,2025-10-01 00:40:09,18119924268,559917769,445,21255,0,12721,9939,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/config.php?id=345569,0,0,benign,1
4,2025-10-01 01:19:57,238229195174,161827224,25167,443,0,3592,1081,Mozilla/5.0 (iPhone; CPU iPhone OS 16_6 like M...,https://internal.bank.local/?id=432022,1,0,benign,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6763,2025-12-29 22:35:29,1003151108,2491798177,44655,3306,0,102224,161909,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://webmail.corp/admin?id=667076,0,0,benign,1
6764,2025-12-29 22:51:57,152953114,236104126125,3306,22,1,244511,280085,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/phpmyadmin?id=120556,1,0,benign,-1
6765,2025-12-29 23:03:05,21354194229,2163437197,3389,445,0,10597,16670,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://app.company.in/api/v1/users?id=525850,0,0,benign,1
6766,2025-12-29 23:20:46,1912213100,243249117224,31882,80,0,6062,3504,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,https://app.company.in/manager/html?id=497838,0,0,benign,1


In [49]:
#skip first
anomalies = df[df["prediction"] == -1]
anomalies

,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,bytes_sent,bytes_received,user_agent,url,is_internal_traffic,label,attack_type,prediction
1,2025-10-01 00:23:43,68592643,2127538111,51165,1433,0,676368,2643374,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/owa/auth/logon.aspx...,0,0,benign,-1
7,2025-10-01 01:57:54,6950130168,20251199195,12037,80,1,136311,418831,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://api.service.io/owa/auth/logon.aspx?id=...,1,0,benign,-1
20,2025-10-01 08:15:43,1019125128,2556120051,21027,53,1,144452,444250,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,https://webmail.corp/phpmyadmin?id=453617,1,0,benign,-1
26,2025-10-01 10:58:59,852001132,14314743154,22,443,1,184486,202730,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://app.company.in/?id=503436,1,0,benign,-1
44,2025-10-01 18:13:44,17918984113,6980117194,34670,64323,2,19715,75045,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://api.service.io/test.php?id=709208,0,0,benign,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6721,2025-12-29 08:39:06,1397712577,3191248193,40954,443,0,944302,3895470,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://api.service.io/owa/auth/logon.aspx?id=...,0,0,benign,-1
6725,2025-12-29 09:33:05,1366820650,1734119446,15894,25,0,440590,1782606,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://webmail.corp/wp-login.php?id=314729,1,0,benign,-1
6730,2025-12-29 10:27:50,18692110162,218752966,32813,21,2,22438,35509,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://internal.bank.local/dashboard?id=910326,1,0,benign,-1
6742,2025-12-29 14:55:02,11451218213,1181120991,22822,1433,0,933798,1347493,zgrab/0.x,https://internal.bank.local/api/v1/users?id=46...,0,1,xss,-1
